# CP5 – Ejercicios extra de PySpark


In [3]:
### Importar las librerías necesarias ###
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
### INICIA LA SPARK-SESSION ###
spark = SparkSession.builder \
    .appName("PySpark - Operaciones avanzadas 2") \
    .getOrCreate()
### Añade el spark context ###
sc = spark.sparkContext
### Añade el path para acceder a los datos y nómbralo como DATA_PATH###
DATA_PATH = "/home/jovyan/data/"

In [4]:
### Lee la carpeta vuelos con pyspark y nómbralo como df_v ###
df_v = spark.read.parquet(DATA_PATH + "vuelos")

## Ejercicio 1 – Aeropuertos con mayor retraso medio

Usando el DataFrame `df_v`, agrupa los vuelos por `ORIGIN_AIRPORT` y calcula:

- El número total de vuelos.
- El retraso medio de llegada (`ARRIVAL_DELAY`).
- El retraso máximo de llegada.

Renombra las columnas resultantes con nombres claros y ordena el resultado de mayor a menor retraso medio.

Muestra únicamente los 10 primeros aeropuertos.

**Funciones a practicar:** `groupBy()`, `agg()`, `count()`, `avg()`, `max()`, `alias()`, `orderBy()`.


In [5]:
retrasos_aeropuerto = (
    df_v.groupBy("ORIGIN_AIRPORT")
    .agg(
        F.count("*").alias("total_vuelos"),
        F.avg("ARRIVAL_DELAY").alias("retraso_medio_llegada"),
        F.max("ARRIVAL_DELAY").alias("retraso_maximo_llegada")
    )
    .orderBy(F.col("retraso_medio_llegada").desc_nulls_last())
)
retrasos_aeropuerto.show(10, truncate=False)

+--------------+------------+---------------------+----------------------+
|ORIGIN_AIRPORT|total_vuelos|retraso_medio_llegada|retraso_maximo_llegada|
+--------------+------------+---------------------+----------------------+
|11905         |2           |390.0                |390                   |
|14674         |2           |169.0                |354                   |
|10333         |1           |98.0                 |98                    |
|12129         |4           |95.0                 |418                   |
|14256         |2           |92.5                 |158                   |
|13541         |1           |43.0                 |43                    |
|ALO           |21          |40.05263157894737    |401                   |
|MEI           |28          |39.964285714285715   |702                   |
|13256         |7           |38.57142857142857    |328                   |
|15841         |2           |35.0                 |81                    |
+--------------+---------

## Ejercicio 2 – Distancia media por mes

Usando `df_v`, agrupa los vuelos por `MONTH` y calcula:

- La distancia media (`DISTANCE`).
- La distancia mínima.
- La distancia máxima.

Ordena el resultado por número de mes de forma ascendente.

**Funciones a practicar:** `groupBy()`, `agg()`, `avg()`, `min()`, `max()`.


In [8]:
distancias_mes = (
    df_v.groupBy("MONTH")
    .agg(
        F.avg("DISTANCE").alias("distancia_media"),
        F.min("DISTANCE").alias("distancia_minima"),
        F.max("DISTANCE").alias("distancia_maxima")
    )
    .orderBy(F.col("MONTH").asc())
)
distancias_mes.show()

+-----+-----------------+----------------+----------------+
|MONTH|  distancia_media|distancia_minima|distancia_maxima|
+-----+-----------------+----------------+----------------+
|    1|806.1322384255982|              67|            4983|
|    2|796.7817524424955|              31|            4962|
|    3|817.7122424878625|              31|            4983|
|    4|823.2305289462724|              31|            4962|
|    5|820.1246904490998|              31|            4983|
|    6|831.4370439788473|              31|            4962|
|    7| 834.798123563952|              31|            4983|
|    8|837.0570454396433|              31|            4983|
|    9|818.4249569954128|              31|            4983|
|   10|820.1984110816867|              31|            4983|
|   11|812.9152984285815|              67|            4983|
|   12|835.3684431389006|              31|            4983|
+-----+-----------------+----------------+----------------+



## Ejercicio 3 – Tabla pivotada de estudiantes

Usando `df_e`, crea una tabla dinámica en la que:

- Las filas representen `graduacion`.
- Las columnas correspondan a los valores de `sexo`.
- Los valores sean el promedio de `peso`.

Después, muestra el resultado.

**Funciones a practicar:** `groupBy()`, `pivot()`, `agg()`, `avg()`.


In [12]:
df_e = spark.read.parquet(DATA_PATH + "estudiantes.parquet")

In [ ]:
import matplotlib.pyplot as plt

In [13]:
peso_por_graduacion_y_sexo = (
    df_e.groupBy("graduacion")
    .pivot("sexo")
    .agg(F.avg("peso"))
    .orderBy("graduacion")
)
peso_por_graduacion_y_sexo.show()

+----------+----+----+
|graduacion|   F|   M|
+----------+----+----+
|      2000|50.0|77.5|
|      2001|65.0|76.0|
+----------+----+----+



## Ejercicio 4 – Empleados con información de departamento

Combina los DataFrames `emple` y `depart` mediante un `inner join`.

La relación debe establecerse entre:

- `emple.num_dpto`
- `depart.id`

Muestra el resultado y comprueba qué información aporta cada DataFrame.

**Funciones a practicar:** `join()`, `col()`.


In [14]:
emple = spark.read.parquet(DATA_PATH + "empleados.parquet")
depart = spark.read.parquet(DATA_PATH + "departamentos.parquet")

In [15]:
empleados_departamentos = emple.alias("e").join(
    depart.alias("d"),
    F.col("e.num_dpto") == F.col("d.id"),
    "inner"
)
empleados_departamentos.show(truncate=False)

# Se mantienen las columnas de ambos DataFrames para poder compararlas.
empleados_departamentos.printSchema()

+------+--------+---+-----------+
|nombre|num_dpto|id |nombre_dpto|
+------+--------+---+-----------+
|Luis  |33      |33 |derecho    |
|Katia |33      |33 |derecho    |
|Raul  |34      |34 |matemática |
|Laura |34      |34 |matemática |
|Sandro|31      |31 |letras     |
+------+--------+---+-----------+

root
 |-- nombre: string (nullable = true)
 |-- num_dpto: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre_dpto: string (nullable = true)



## Ejercicio 5 – Departamentos sin empleados

Obtén únicamente los departamentos que no tienen ningún empleado asociado.

Parte de los DataFrames `depart` y `emple` y utiliza el tipo de join adecuado para obtener solo los registros de `depart` que no tienen correspondencia en `emple`.

**Funciones a practicar:** `join()` y `left anti join`.


In [ ]:
departamentos_sin_empleados = depart.alias("d").join(
    emple.alias("e"),
    F.col("d.id") == F.col("e.num_dpto"),
    "left_anti"
)
departamentos_sin_empleados.show(truncate=False)

## Ejercicio 6 – Antigüedad de los registros

Usando `df_calc`, crea un nuevo DataFrame que muestre:

- `nombre`
- `fecha_ingreso`
- `fecha_salida`
- El número de días transcurridos entre ambas fechas.
- El número aproximado de meses transcurridos entre ambas fechas.

Renombra las dos columnas calculadas como `dias_transcurridos` y `meses_transcurridos`.

**Funciones a practicar:** `datediff()`, `months_between()`, `alias()`.


In [17]:
df_calc = spark.read.parquet(DATA_PATH + "calculo.parquet")

In [ ]:
antiguedad = df_calc.select(
    "nombre",
    "fecha_ingreso",
    "fecha_salida",
    F.datediff(F.col("fecha_salida"), F.col("fecha_ingreso")).alias("dias_transcurridos"),
    F.months_between(F.col("fecha_salida"), F.col("fecha_ingreso")).alias("meses_transcurridos")
)
antiguedad.show(truncate=False)

## Ejercicio 7 – Descomponer una fecha y hora

Usando la columna `baja_sistema` de `df_calc`, crea una selección que incluya:

- Año.
- Mes.
- Día del mes.
- Día del año.
- Hora.
- Minuto.
- Segundo.

Asigna un alias descriptivo a cada columna.

**Funciones a practicar:** `year()`, `month()`, `dayofmonth()`, `dayofyear()`, `hour()`, `minute()`, `second()`.


In [ ]:
componentes_fecha = df_calc.select(
    F.year("baja_sistema").alias("anio"),
    F.month("baja_sistema").alias("mes"),
    F.dayofmonth("baja_sistema").alias("dia_mes"),
    F.dayofyear("baja_sistema").alias("dia_anio"),
    F.hour("baja_sistema").alias("hora"),
    F.minute("baja_sistema").alias("minuto"),
    F.second("baja_sistema").alias("segundo")
)
componentes_fecha.show()

## Ejercicio 8 – Normalización de texto

Crea un DataFrame con una columna llamada `texto` que contenga varias cadenas con espacios al principio y al final.

A partir de esa columna, genera:

- El texto sin espacios exteriores.
- El texto en minúsculas.
- El texto en mayúsculas.
- El texto con la primera letra de cada palabra en mayúscula.
- El texto invertido.

**Funciones a practicar:** `trim()`, `lower()`, `upper()`, `initcap()`, `reverse()`.


In [ ]:
textos = spark.createDataFrame(
    [("  Hola Mundo  ",), ("  PySpark es útil  ",), ("  análisis DE DATOS  ",)],
    ["texto"]
)

textos_normalizados = textos.select(
    "texto",
    F.trim("texto").alias("sin_espacios_exteriores"),
    F.lower(F.trim("texto")).alias("minusculas"),
    F.upper(F.trim("texto")).alias("mayusculas"),
    F.initcap(F.trim("texto")).alias("tipo_titulo"),
    F.reverse(F.trim("texto")).alias("invertido")
)
textos_normalizados.show(truncate=False)

## Ejercicio 9 – Analizar colecciones de tareas

Usando `df_c`, crea una transformación que muestre para cada `dia`:

- El número de elementos del array `tareas`.
- El array ordenado.
- Si el array contiene la tarea `buscar agua`.

Después, utiliza `explode()` para crear una segunda vista en la que cada tarea aparezca en una fila independiente.

**Funciones a practicar:** `size()`, `sort_array()`, `array_contains()`, `explode()`.


In [ ]:
resumen_tareas = df_c.select(
    "dia",
    F.size("tareas").alias("numero_tareas"),
    F.sort_array("tareas").alias("tareas_ordenadas"),
    F.array_contains("tareas", "buscar agua").alias("contiene_buscar_agua")
)
resumen_tareas.show(truncate=False)

# Segunda vista: una fila por cada tarea de cada día.
tareas_por_fila = df_c.select(
    "dia",
    F.explode("tareas").alias("tarea")
)
tareas_por_fila.show(truncate=False)

## Ejercicio 10 – Clasificar y completar datos

Usando `df_p`, crea un nuevo DataFrame que:

1. Sustituya los valores nulos de `nombre` por el texto `sin nombre`.
2. Cree una columna `estado_pago` con estas reglas:
   - Si `pago == 1`: `pagado`.
   - Si `pago == 2`: `pendiente`.
   - Para cualquier otro valor: `sin iniciar`.
3. Muestre únicamente `nombre` y `estado_pago`.

**Funciones a practicar:** `coalesce()`, `lit()`, `when()`, `otherwise()`.


In [ ]:
pagos_clasificados = (
    df_p.select(
        F.coalesce(F.col("nombre"), F.lit("sin nombre")).alias("nombre"),
        F.when(F.col("pago") == 1, F.lit("pagado"))
         .when(F.col("pago") == 2, F.lit("pendiente"))
         .otherwise(F.lit("sin iniciar"))
         .alias("estado_pago")
    )
)
pagos_clasificados.show(truncate=False)